# Notebook 007 — Genes Hub: Consenso e Cruzamento com Enriquecimento

Identifica os genes hub consensuais (presentes em ambas as métricas MCC e Degree
no ranking TOP 10 da rede STRING High Confidence) e verifica quais termos
de enriquecimento funcional (notebook 006) são dirigidos por esses hubs.

**Entradas:**
- `network_export/hubs_high_mcc_top10/20.csv` — hubs por MCC (cytoHubba)
- `network_export/hubs_high_degree_top10/20.csv` — hubs por Degree (cytoHubba)
- `enrichment/{module}_enrichr_results.csv` — resultados ORA por módulo (notebook 006)

**Referência:** Sainz et al. (2024) usa kWithin (WGCNA); aqui usamos MCC/Degree
sobre a rede PPI STRING — abordagem padrão para redes proteína-proteína.

## Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

## Configurações

In [ ]:
NETWORK_DIR    = Path("../../data/processed/network_export")
ENRICHMENT_DIR = Path("../../data/processed/enrichment")
HUB_DIR        = NETWORK_DIR

# Módulos com enriquecimento significativo (FDR < 0.05)
ENRICHED_MODULES = ["dimgrey"]  # white e indianred têm < 10 genes — tratados como exploratório

## Carregamento e Consenso dos Hubs

In [ ]:
KEEP_COLS = ["gene_symbol", "gene_name", "primary_module",
             "strongest_contrast", "strongest_regulation",
             "strongest_log2FoldChange", "strongest_padj",
             "n_contrasts", "trait_support"]

def load_hub(path):
    df = pd.read_csv(path)
    return df[[c for c in KEEP_COLS if c in df.columns]].copy()

mcc10    = load_hub(HUB_DIR / "hubs_high_mcc_top10.csv")
mcc20    = load_hub(HUB_DIR / "hubs_high_mcc_top20.csv")
degree10 = load_hub(HUB_DIR / "hubs_high_degree_top10.csv")
degree20 = load_hub(HUB_DIR / "hubs_high_degree_top20.csv")

# União: concat todos, desduplicar para metadata, depois adicionar indicadores
all_hubs = (
    pd.concat([mcc10, mcc20, degree10, degree20])
    .drop_duplicates(subset="gene_symbol")
    .reset_index(drop=True)
)

all_hubs["in_mcc_10"]    = all_hubs["gene_symbol"].isin(mcc10["gene_symbol"])
all_hubs["in_mcc_20"]    = all_hubs["gene_symbol"].isin(mcc20["gene_symbol"])
all_hubs["in_degree_10"] = all_hubs["gene_symbol"].isin(degree10["gene_symbol"])
all_hubs["in_degree_20"] = all_hubs["gene_symbol"].isin(degree20["gene_symbol"])

all_hubs["n_lists"] = (
    all_hubs["in_mcc_10"].astype(int) +
    all_hubs["in_mcc_20"].astype(int) +
    all_hubs["in_degree_10"].astype(int) +
    all_hubs["in_degree_20"].astype(int)
)

all_hubs = all_hubs.sort_values("n_lists", ascending=False).reset_index(drop=True)

print(f"Total de genes hub únicos: {len(all_hubs)}")
print(f"Consenso absoluto (4/4 listas): {(all_hubs['n_lists'] == 4).sum()} genes")
print(f"Consenso MCC∩Degree TOP 10:     {(all_hubs['in_mcc_10'] & all_hubs['in_degree_10']).sum()} genes")

display(all_hubs[["gene_symbol", "gene_name", "primary_module",
                  "strongest_contrast", "strongest_regulation",
                  "in_mcc_10", "in_degree_10", "in_mcc_20", "in_degree_20", "n_lists"]])

## Hubs Consensuais (MCC ∩ Degree TOP 10)

In [ ]:
consensus_hubs = all_hubs[all_hubs["in_mcc_10"] & all_hubs["in_degree_10"]].copy()
consensus_symbols = set(consensus_hubs["gene_symbol"])

print(f"Hubs consensuais (MCC ∩ Degree TOP 10): {len(consensus_hubs)}")
print(consensus_hubs["gene_symbol"].tolist())

# Módulos representados
print("\nMódulos:")
print(consensus_hubs.groupby("primary_module")["gene_symbol"].apply(list))

# Direção da regulação
print("\nRegulação no contraste mais forte:")
display(consensus_hubs[["gene_symbol", "strongest_contrast",
                         "strongest_regulation", "strongest_log2FoldChange",
                         "strongest_padj"]].reset_index(drop=True))

## Cruzamento com Enriquecimento Funcional

Para cada termo significativo (FDR < 0.05) nos módulos enriquecidos,
verifica quais genes hub do consenso estão presentes na lista de overlap.

In [ ]:
crossref_rows = []

for module in ENRICHED_MODULES:
    enr_path = ENRICHMENT_DIR / f"{module}_enrichr_results.csv"
    if not enr_path.exists():
        print(f"[{module}] arquivo não encontrado")
        continue

    enr = pd.read_csv(enr_path)
    sig = enr[enr["Adjusted P-value"] < 0.05].copy()

    for _, row in sig.iterrows():
        term_genes  = set(str(row["Genes"]).split(";")) if pd.notna(row["Genes"]) else set()
        hub_overlap = sorted(consensus_symbols & term_genes)
        crossref_rows.append({
            "module":           module,
            "Gene_set":         row["Gene_set"],
            "Term":             row["Term"],
            "Adjusted P-value": row["Adjusted P-value"],
            "n_term_genes":     len(term_genes),
            "n_hub_overlap":    len(hub_overlap),
            "hub_genes":        ";".join(hub_overlap),
        })

crossref = pd.DataFrame(crossref_rows).sort_values(
    ["module", "Gene_set", "Adjusted P-value"]
).reset_index(drop=True)

print(f"Termos com ≥ 1 gene hub no overlap: {(crossref['n_hub_overlap'] > 0).sum()} / {len(crossref)}")
display(crossref)

## Visualização — Heatmap Hub × Termo

In [ ]:
# Foco: termos GO BP do módulo com hubs
for module in ENRICHED_MODULES:
    sub = crossref[
        (crossref["module"] == module) &
        (crossref["Gene_set"] == "GO_Biological_Process_2023") &
        (crossref["n_hub_overlap"] > 0)
    ].copy()

    if sub.empty:
        print(f"[{module}] Nenhum termo GO BP com hub no overlap.")
        continue

    # Matriz binária: linhas = termos, colunas = hub genes
    hub_cols = sorted(consensus_symbols)
    matrix   = []
    term_labels = []

    for _, row in sub.iterrows():
        hubs_in_term = set(str(row["hub_genes"]).split(";"))
        matrix.append([1 if g in hubs_in_term else 0 for g in hub_cols])
        label = row["Term"]
        label = (label[:55] + "…") if len(label) > 57 else label
        term_labels.append(f"{label}  (padj={row['Adjusted P-value']:.2e})")

    mat_df = pd.DataFrame(matrix, index=term_labels, columns=hub_cols)

    fig = go.Figure(go.Heatmap(
        z=mat_df.values,
        x=mat_df.columns.tolist(),
        y=mat_df.index.tolist(),
        colorscale=[[0, "#f0f0f0"], [1, "#2166ac"]],
        showscale=False,
        hovertemplate="%{y}<br>%{x}<br>presente: %{z}<extra></extra>",
    ))
    fig.update_layout(
        title=f"{module} — Hubs Consensuais × Termos GO BP Significativos",
        xaxis=dict(tickangle=-45, title="Gene Hub"),
        yaxis=dict(autorange="reversed", title=""),
        height=max(300, 28 * len(term_labels) + 150),
        width=900,
        margin=dict(l=400, r=40, t=60, b=100),
    )
    fig.show()

## Nota sobre white e indianred

Os módulos **white** (8 genes DEG) e **indianred** (6 genes) apresentaram
respectivamente 20 e 34 termos GO BP significativos, mas com listas de entrada
muito pequenas. Com tão poucos genes, a ORA tem baixo poder estatístico e
alta sensibilidade a coincidências — os resultados devem ser tratados como
exploratórios e não conclusivos.

## Resumo Final

In [ ]:
print("=" * 62)
print("RESUMO — GENES HUB E ENRIQUECIMENTO FUNCIONAL")
print("=" * 62)

print(f"\nHubs consensuais (MCC ∩ Degree TOP 10, HIGH confidence):")
for g in sorted(consensus_symbols):
    row = consensus_hubs[consensus_hubs["gene_symbol"] == g].iloc[0]
    print(f"  {g:12s}  módulo={row.get('primary_module','?')}  "
          f"contraste={row.get('strongest_contrast','?')}  "
          f"reg={row.get('strongest_regulation','?')}  "
          f"log2FC={row.get('strongest_log2FoldChange', float('nan')):.2f}")

print("\nTermos GO BP do dimgrey dirigidos pelos hubs:")
for _, row in crossref[
    (crossref["module"] == "dimgrey") &
    (crossref["Gene_set"] == "GO_Biological_Process_2023") &
    (crossref["n_hub_overlap"] > 0)
].iterrows():
    print(f"  [{row['n_hub_overlap']}/{row['n_term_genes']} hubs] "
          f"padj={row['Adjusted P-value']:.2e}  {row['Term']}")

print("\nTermos KEGG do dimgrey dirigidos pelos hubs:")
for _, row in crossref[
    (crossref["module"] == "dimgrey") &
    (crossref["Gene_set"] == "KEGG_2021_Human") &
    (crossref["n_hub_overlap"] > 0)
].iterrows():
    print(f"  [{row['n_hub_overlap']}/{row['n_term_genes']} hubs] "
          f"padj={row['Adjusted P-value']:.2e}  {row['Term']}")

In [ ]:
consensus_hubs.to_csv(NETWORK_DIR / "hub_consensus.csv", index=False)
crossref.to_csv(ENRICHMENT_DIR / "hub_enrichment_crossref.csv", index=False)

print("Saved:")
print(f"  {NETWORK_DIR / 'hub_consensus.csv'}")
print(f"  {ENRICHMENT_DIR / 'hub_enrichment_crossref.csv'}")